# 00 — Data audit

**Spec: §4.** Run this before touching a model. Everything downstream inherits
whatever is wrong here, and most of the checks take under ten minutes.

Produces four things the rest of the project depends on:

| Output | Consumed by |
|---|---|
| The polarity-hole finding | §9.2 neutral band, README |
| Duplicate / leakage report | Final test table |
| `max_length` + truncation rate | `inference_config.json` |
| 40k stratified subsample | Every training notebook |

**Done when** all four are recorded and `inference_config.json` has
`max_length` filled in.

In [ ]:
# Kaggle/Colab setup. Skip if running locally with the repo already present.
# !pip install -q transformers datasets sentencepiece protobuf

import sys, re, json
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
from src.normalize import normalize, NORMALIZE_VERSION

print("python", sys.version.split()[0], "| normalize", NORMALIZE_VERSION)

## 4.1 Confirm the dataset and its construction

`allocine` — ~200k real French reviews, binary sentiment, 160k/20k/20k.
Real human-written French with real labels: this is what the pivot buys.

**Verify shape on day 1** rather than trusting the spec — Hub datasets move
and get deprecated. Fallback if unavailable: `cardiffnlp/tweet_sentiment_multilingual`
(French config, 3-class, smaller).

In [ ]:
from datasets import load_dataset

ds = load_dataset("allocine")
print(ds)

TEXT_COL = "review" if "review" in ds["train"].column_names else "text"
LABEL_COL = "label"
print("text column:", TEXT_COL)
print(ds["train"].features)

In [ ]:
# Class balance. Allocine should be near-balanced -- that is what lets us
# skip the class-weighted loss the emotion version needed.
for split in ds:
    c = Counter(ds[split][LABEL_COL])
    total = sum(c.values())
    print(f"{split:12s} n={total:>6d}  " +
          "  ".join(f"{k}:{v} ({v/total:.1%})" for k, v in sorted(c.items())))

### The finding to look for: the polarity hole

Review datasets built from star ratings are usually constructed by taking high
ratings as positive, low as negative, and **discarding the middle**.

If that happened here, **neutral text is out-of-distribution by construction** —
the training data has a hole exactly where real client feedback is densest.
That is not a flaw you can train away. It is the direct justification for the
neutral band (§9.2) and for the domain-shift test (§6).

Read 15 examples per class and judge: are there any genuinely *lukewarm*
reviews, or only strong opinions?

In [ ]:
rng = np.random.default_rng(0)
for label in sorted(set(ds["train"][LABEL_COL])):
    idx = [i for i, l in enumerate(ds["train"][LABEL_COL][:20000]) if l == label]
    print(f"\n{'='*70}\nLABEL {label}\n{'='*70}")
    for i in rng.choice(idx, 15, replace=False):
        t = ds["train"][TEXT_COL][int(i)]
        print(f"  - {t[:180]}{'...' if len(t) > 180 else ''}")

In [ ]:
# VERDICT -- write your judgement here; it goes in the README.
POLARITY_HOLE = None   # True if lukewarm reviews are absent by construction
POLARITY_NOTE = """
<Replace: did you see any 3-star-equivalent, lukewarm, or mixed reviews?
 Or only strong opinions in both directions?>
"""
print(POLARITY_NOTE)

## 4.2 Duplicate leakage

Scraped review corpora routinely contain duplicates (reposts, bot reviews,
template text). Any text in both train and test inflates your score for free.

The `conflict` count is a bonus: identical text with two different gold labels
is an irreducible error floor, and tells you how noisy the labels are.

In [ ]:
def norm_key(t):
    import unicodedata
    t = unicodedata.normalize("NFKC", t).lower().strip()
    return re.sub(r"\s+", " ", t)

seen = defaultdict(list)
for split in ds:
    for i, t in enumerate(ds[split][TEXT_COL]):
        seen[norm_key(t)].append((split, i))

cross  = {k: v for k, v in seen.items() if len({s for s, _ in v}) > 1}
within = {k: v for k, v in seen.items() if len(v) > 1 and len({s for s, _ in v}) == 1}

print(f"unique texts:              {len(seen):>7d}")
print(f"texts in >1 split:         {len(cross):>7d}")
print(f"duplicated within a split: {len(within):>7d}")

In [ ]:
# Identical texts carrying conflicting labels = irreducible error floor.
label_lookup = {split: ds[split][LABEL_COL] for split in ds}
conflict = [v for v in seen.values()
            if len(v) > 1 and len({label_lookup[s][i] for s, i in v}) > 1]

print(f"identical texts, conflicting labels: {len(conflict)}")
if conflict:
    ceiling = 1 - len(conflict) / sum(len(ds[s]) for s in ds)
    print(f"implied accuracy ceiling: ~{ceiling:.4f}")
    for v in conflict[:5]:
        s, i = v[0]
        print(f"  - {ds[s][TEXT_COL][i][:150]}")

In [ ]:
# Build a deduplicated test index. Report BOTH numbers in the final table --
# do not silently drop rows.
train_val_keys = {norm_key(t) for s in ("train", "validation") if s in ds
                  for t in ds[s][TEXT_COL]}
test_clean_idx = [i for i, t in enumerate(ds["test"][TEXT_COL])
                  if norm_key(t) not in train_val_keys]

print(f"test rows: {len(ds['test'])} -> {len(test_clean_idx)} after dedup "
      f"({len(ds['test']) - len(test_clean_idx)} removed)")
np.save(REPO / "data" / "test_clean_idx.npy", np.array(test_clean_idx))

## 4.3 Length, and the subsample decision

**This is a budget decision, not a detail.** Allociné reviews are long. At
160k rows × 256 tokens, one epoch on a T4 runs into hours and the three-seed
requirement (§1.3) becomes unaffordable — which would quietly pressure you into
dropping the seeds, the one thing you must not drop.

Target: **40k stratified train rows**, `max_length` near p90.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("almanach/camembert-base")

sample = ds["train"][TEXT_COL][:5000]
lens = np.array([len(tok(normalize(t)).input_ids) for t in sample])

for q in (50, 75, 90, 95, 99):
    print(f"p{q:<3d} {np.percentile(lens, q):>6.0f} tokens")
print(f"max  {lens.max():>6d} tokens")

plt.figure(figsize=(8, 3))
plt.hist(lens, bins=80)
plt.axvline(np.percentile(lens, 90), color="crimson", ls="--", label="p90")
plt.xlabel("tokens"); plt.ylabel("reviews"); plt.legend()
plt.title("Token length distribution (CamemBERT tokenizer)")
plt.tight_layout(); plt.show()

In [ ]:
MAX_LENGTH = int(np.ceil(np.percentile(lens, 90) / 32) * 32)  # round to /32
trunc_rate = (lens > MAX_LENGTH).mean()

print(f"MAX_LENGTH = {MAX_LENGTH}")
print(f"truncation rate: {trunc_rate:.1%}")
print()
print("NOTE for the limitations section: review sentiment often lands in the")
print("final sentence, so truncation is not a neutral loss. Say so if >10%.")

In [ ]:
# 40k stratified subsample. Fixed seed -- every training notebook must load
# exactly this subset or the three-seed comparison is not controlled.
N_SUB = 40_000
train = ds["train"]
labels = np.array(train[LABEL_COL])

rng = np.random.default_rng(42)
per_class = N_SUB // len(set(labels))
sub_idx = np.concatenate([
    rng.choice(np.flatnonzero(labels == c), per_class, replace=False)
    for c in sorted(set(labels))
])
rng.shuffle(sub_idx)

np.save(REPO / "data" / "train_subsample_40k.npy", sub_idx)
print(f"subsample: {len(sub_idx)} rows, balance = "
      f"{dict(Counter(labels[sub_idx]))}")

## 4.4 Text format and `normalize()`

Whatever formatting the corpus has is baked into your model. If the Space feeds
it a different format, part of your measured drop is formatting, not language —
and you would misattribute it.

`src/normalize.py` v1 deliberately **preserves** casing, punctuation, accents
and emoji, because all of them carry sentiment signal in reviews. If the audit
below shows the corpus is already lowercased, set `LOWERCASE = True` there and
**bump `NORMALIZE_VERSION`** (the config validator will then force a
re-calibration rather than letting stale thresholds pair with new code).

In [ ]:
checks = {
    "has uppercase":   lambda t: t != t.lower(),
    "has punctuation": lambda t: bool(re.search(r"[.,!?;:]", t)),
    "has apostrophe":  lambda t: "'" in t or "\u2019" in t,
    "has emoji":       lambda t: bool(re.search(r"[\U0001F300-\U0001FAFF]", t)),
    "has html":        lambda t: bool(re.search(r"<[^>]+>", t)),
    "has url":         lambda t: bool(re.search(r"https?://", t)),
}
for name, fn in checks.items():
    print(f"{name:<18s} {sum(fn(t) for t in sample) / len(sample):>7.1%}")

In [ ]:
# Sanity-check normalize() on real rows before committing to it.
for t in sample[:5]:
    print("RAW :", t[:120])
    print("NORM:", normalize(t)[:120])
    print()

## Write results to `inference_config.json`

`max_length` and `normalize_version` are this notebook's deliverables. The
remaining fields stay `null` until their own phases fill them — `load_config()`
will name the responsible notebook if anything tries to run early.

In [ ]:
cfg_path = REPO / "inference_config.json"
cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
cfg["max_length"] = MAX_LENGTH
cfg["normalize_version"] = NORMALIZE_VERSION
cfg_path.write_text(json.dumps(cfg, indent=2, ensure_ascii=False) + "\n",
                    encoding="utf-8")
print(cfg_path.read_text(encoding="utf-8"))

## Audit summary — paste into the README

Fill each blank from the cells above. These are findings, not statistics:
every one of them is a sentence a reader will remember.

In [ ]:
print(f"""
AUDIT SUMMARY
=============
Dataset            : allocine, {sum(len(ds[s]) for s in ds):,} rows total
Class balance      : {dict(Counter(ds['train'][LABEL_COL]))}
Polarity hole      : {POLARITY_HOLE}  <- set this manually above
Cross-split dupes  : {len(cross)}
Conflicting labels : {len(conflict)}
max_length         : {MAX_LENGTH} (p90), truncation {trunc_rate:.1%}
Train subsample    : {len(sub_idx):,} rows, seed 42
normalize version  : {NORMALIZE_VERSION}
""")